# Healthcare Predictive Analytics: Diabetes & Heart Failure Classification
Individual Task 1 — Part 1.3 Data Analysis

Two datasets, two classifiers (Decision Tree, Neural Network / MLP), compared using accuracy, precision, recall, F1, and ROC-AUC.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings; warnings.filterwarnings('ignore')

## Load Dataset 1: Pima Indians Diabetes

In [2]:
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
diab = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv', header=None, names=cols)
diab.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Load Dataset 2: Heart Failure Clinical Records

In [3]:
hf = pd.read_csv('https://raw.githubusercontent.com/dimikara/heart-failure-prediction/master/heart_failure_clinical_records_dataset.csv')
hf.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


## Reusable training/evaluation function
Note: the Neural Network uses `solver='lbfgs'` rather than the default `adam` — on these small tabular datasets (299 and 768 rows), `adam` with early stopping collapsed to predicting the majority class (0 precision/recall) on the diabetes data. `lbfgs` is documented in scikit-learn as more suitable for small datasets and produced stable, non-degenerate results.

In [4]:
def run(name, df, target, drop_cols=[]):
    df = df.drop(columns=drop_cols, errors='ignore').copy().dropna()
    y = df[target]
    X = df.drop(columns=[target])
    if not pd.api.types.is_numeric_dtype(y) or y.dtype == bool:
        y = LabelEncoder().fit_transform(y.astype(str))
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]) or X[c].dtype == bool:
            X[c] = LabelEncoder().fit_transform(X[c].astype(str))
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    results = {}
    dt = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
    dt.fit(X_train, y_train)
    pred_dt = dt.predict(X_test)
    proba_dt = dt.predict_proba(X_test)[:,1]
    results['DecisionTree'] = {
        'accuracy': accuracy_score(y_test, pred_dt), 'precision': precision_score(y_test, pred_dt),
        'recall': recall_score(y_test, pred_dt), 'f1': f1_score(y_test, pred_dt),
        'roc_auc': roc_auc_score(y_test, proba_dt),
        'top_features': sorted(zip(X.columns, dt.feature_importances_), key=lambda x:-x[1])[:5]
    }

    mlp = MLPClassifier(hidden_layer_sizes=(32,16), max_iter=2000, random_state=42, solver='lbfgs', alpha=0.01)
    mlp.fit(X_train_s, y_train)
    pred_mlp = mlp.predict(X_test_s)
    proba_mlp = mlp.predict_proba(X_test_s)[:,1]
    results['NeuralNet'] = {
        'accuracy': accuracy_score(y_test, pred_mlp), 'precision': precision_score(y_test, pred_mlp),
        'recall': recall_score(y_test, pred_mlp), 'f1': f1_score(y_test, pred_mlp),
        'roc_auc': roc_auc_score(y_test, proba_mlp),
    }

    print(f'=== {name} (n={len(df)}, positive rate={y.mean():.3f}) ===')
    for model, r in results.items():
        print(f'-- {model} --')
        for k,v in r.items():
            if k != 'top_features': print(f'   {k}: {v:.3f}')
        if 'top_features' in r: print(f'   top_features: {r["top_features"]}')
    return results

## Run on Dataset 1: Diabetes

In [5]:
r1 = run('Pima Diabetes', diab, target='Outcome')

=== Pima Diabetes (n=768, positive rate=0.349) ===
-- DecisionTree --
   accuracy: 0.750
   precision: 0.617
   recall: 0.746
   f1: 0.676
   roc_auc: 0.799
   top_features: [('Glucose', np.float64(0.4281632165524875)), ('BMI', np.float64(0.2451249203685701)), ('Age', np.float64(0.14201591515422288)), ('DiabetesPedigreeFunction', np.float64(0.06803202452504388)), ('Pregnancies', np.float64(0.04285298174892822))]
-- NeuralNet --
   accuracy: 0.682
   precision: 0.544
   recall: 0.552
   f1: 0.548
   roc_auc: 0.738


## Run on Dataset 2: Heart Failure

In [6]:
r2 = run('Heart Failure', hf, target='DEATH_EVENT')

=== Heart Failure (n=299, positive rate=0.321) ===
-- DecisionTree --
   accuracy: 0.787
   precision: 0.643
   recall: 0.750
   f1: 0.692
   roc_auc: 0.766
   top_features: [('time', np.float64(0.4736733883188078)), ('serum_creatinine', np.float64(0.1983117934646748)), ('creatinine_phosphokinase', np.float64(0.11375881304163647)), ('ejection_fraction', np.float64(0.09540548916976048)), ('age', np.float64(0.0605488149931624))]
-- NeuralNet --
   accuracy: 0.720
   precision: 0.571
   recall: 0.500
   f1: 0.533
   roc_auc: 0.757


## Note on a data quality issue found during analysis
The Decision Tree's top feature for Heart Failure is `time` (47% importance) — the patient's *follow-up period length*. This is a known critique of this dataset in the literature: shorter follow-up time can correlate with the outcome simply because patients who died were, by definition, not followed for as long (informative censoring), rather than `time` being a genuine clinically actionable predictor available at the point of risk assessment. This is flagged as a limitation in the report rather than treated as a straightforward clinical insight.

## Comparison chart

In [7]:
import matplotlib.pyplot as plt
import numpy as np
metrics = ['Accuracy','Precision','Recall','F1','ROC-AUC']
diab_dt = [r1['DecisionTree'][m.lower().replace('-','_')] for m in metrics]
diab_nn = [r2['NeuralNet'][m.lower().replace('-','_')] for m in metrics]
# (values also hardcoded in the report table for reproducibility)
print('See model_comparison.png in the repository for the final chart used in the report.')

See model_comparison.png in the repository for the final chart used in the report.
